# Configurações

## Pacotes

### Desinstalar

In [ ]:
!pip uninstall -y tensorflow

Found existing installation: tensorflow 2.12.0
Uninstalling tensorflow-2.12.0:
  Successfully uninstalled tensorflow-2.12.0


### Instalar

In [ ]:
!pip install tensorflow

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Using cached tensorflow-2.12.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (585.9 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 17.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.3
    Uninstalling numpy-1.24.3:
      Successfully uninstalled numpy-1.24.3


In [ ]:
!pip install osmnx

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
!pip install bridson

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
!pip install rasterstats

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 76.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 67.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.9/137.9 kB 11.3 MB/s eta 0:00:00
  Attempting uninstall: fiona
    Found existing installation: Fiona 1.9.3
    Uninstalling Fiona-1.9.3:
      Successfully uninstalled Fiona-1.9.3


In [ ]:
! pip install haversine

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


### Chamar

In [ ]:
from google.colab import drive
from google.colab import files

import osmnx as ox

from bridson import poisson_disc_samples
import geopandas
from shapely.geometry import Point

import pandas as pd
from rasterstats import  point_query
import numpy as np

import haversine as hs
from haversine import  Unit
import copy

from shapely.ops import nearest_points
from geopy.distance import distance

from shapely import wkt

import random
import numpy as np

# Set a random seed
random.seed(42)
np.random.seed(42)

## Funções

In [ ]:
# Sorteio de coordenadas dentro do polígono

def Poisson_Disc_Random_Points_in_Polygon(polygon, radius_value):

    points = []

    minx = polygon.bounds.loc[0][0]
    miny = polygon.bounds.loc[0][1]
    maxx = polygon.bounds.loc[0][2]
    maxy = polygon.bounds.loc[0][3]

    width = maxx - minx
    height = maxy - miny

    pds = poisson_disc_samples(width=width,
                               height=height,
                               r= radius_value #(0.00125) aproximadamente 100 metros
                               )

    for i in range(len(pds)):
      pnt = Point(tuple(map(sum, zip(pds[i], (minx,miny)))))
      if (polygon.contains(pnt)).bool():
          points.append(pnt)

    return geopandas.GeoSeries(points)

In [ ]:
# Limpeza para pontos apenas dentro do município de Niterói

def Clear_Outbounds_Points_in_Polygon(polygon, pnts):

    points = []

    l = len(pnts)

    for i in range(l):
      if (polygon.contains(pnts[i])).bool():
            points.append(pnts[i])
    return geopandas.GeoSeries(points)

In [ ]:
# Atribuição do n_coord mais próximo

def atribuir_n_coord_e_dist(df_ocorrencia, df_coord):

    coord_dict = dict(zip(df_coord.geometry, df_coord.n_coord))

    df_ocorrencia['dist_to_nearest'] = None

    for index, row in df_ocorrencia.iterrows():

        nearest = nearest_points(row.geometry, df_coord.geometry.unary_union)[1]

        n_coord = coord_dict[nearest]

        dist = distance(row.geometry.coords[0], nearest.coords[0]).meters

        df_ocorrencia.loc[index, 'n_coord'] = n_coord
        df_ocorrencia.loc[index, 'dist_to_nearest'] = dist

    return df_ocorrencia

In [ ]:
#Distância mínima das ocorrências
def Suitable_Points_for_not_Occurence(points_disc_samples, points_occurence, radius ):

  aux1=[list((points_disc_samples[row_id].x,points_disc_samples[row_id].y)) for row_id in range(points_disc_samples.shape[0])]

  for i in range(len(points_occurence)):
    #print(f'')
    aux2=[]
    for j in range(len(aux1)):
      print(f'Amostra, {j} de {len(aux1)} - Ocorrência, {i} de {len(points_occurence)}')
      distancia = hs.haversine((points_occurence[i].x,points_occurence[i].y),
             aux1[j],
             unit=Unit.METERS)
      if distancia > radius:
        aux2.append(aux1[j])
    aux1 = copy.deepcopy(aux2)

  aux2=[]

  for i in range(len(aux1)):
    aux2.append(Point(aux1[i]))

  return geopandas.GeoSeries(aux2)

## Conexão com a pasta

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/My Drive/PDPA

/content/drive/My Drive/PDPA


# Seleção do município de Niterói

In [ ]:
niteroi_q = "Niterói, Rio de Janeiro, Brazil"

In [ ]:
niteroi = ox.geocode_to_gdf(niteroi_q)

In [ ]:
 #niteroi.crs

# Criação de amostras de coordenadas

## Seleção das coordenadas



### Amostra de coordenadas dentro de Niterói

In [ ]:
pts_s_ocorrencia = Poisson_Disc_Random_Points_in_Polygon(niteroi,(0.00125))  # aproximadamente 100 metros de raio

### Retirar área de lagos

In [ ]:
gdf_pedologia = geopandas.read_file("pedo_area_mu_3303302.shp")

In [ ]:
lagos = gdf_pedologia[gdf_pedologia.loc[:,'legenda' ] == 'Corpo d\'água continental'].unary_union

In [ ]:
l = len(pts_s_ocorrencia)
polygon = lagos
pnts = pts_s_ocorrencia
points = []

for i in range(l):
  if (polygon.contains(pnts[i])):
    continue
  else:
    points.append(pnts[i])

In [ ]:
pts_s_ocorrencia = geopandas.GeoSeries(points)

### Retirar regiões a nível do mar

In [ ]:
pts_s_ocorrencia_lista=[list((pts_s_ocorrencia[row_id].x,pts_s_ocorrencia[row_id].y)) for row_id in range(pts_s_ocorrencia.shape[0])]

In [ ]:
df_s_ocorrencia = pd.DataFrame(pts_s_ocorrencia_lista, columns=['lon_ocr','lat_ocr'])
df_s_ocorrencia['geometry'] = pts_s_ocorrencia

In [ ]:
df_s_ocorrencia.loc[:,'Altitude_numerica']= point_query(df_s_ocorrencia.loc[:,'geometry'], '22S435ZN.tif')

In [ ]:
print(min(df_s_ocorrencia['Altitude_numerica']))
print(max(df_s_ocorrencia['Altitude_numerica']))
print(np.mean(df_s_ocorrencia['Altitude_numerica']))

#https://pt-br.topographic-map.com/map-p5151/Niter%C3%B3i/?center=-22.90771%2C-43.10877&zoom=14

0.013144538037607765
375.2385198609718
86.2543342646348


In [ ]:
df_s_ocorrencia_filt = df_s_ocorrencia[df_s_ocorrencia.loc[:,'Altitude_numerica' ] > 25]

In [ ]:
pts_s_ocorrencia_filt = geopandas.GeoSeries(list(df_s_ocorrencia_filt['geometry']))

### Criar chave para cada coordenada

In [ ]:
amostra_aleatoria_coord_100m = geopandas.GeoDataFrame(geometry=pts_s_ocorrencia_filt, crs='EPSG:4326')

In [ ]:
amostra_aleatoria_coord_100m['n_coord'] = range(0,len(amostra_aleatoria_coord_100m))

In [ ]:
amostra_aleatoria_coord_100m

,geometry,n_coord
0,POINT (-43.08662 -22.98827),0
1,POINT (-43.08296 -22.98761),1
2,POINT (-43.08446 -22.98700),2
3,POINT (-43.08222 -22.98648),3
4,POINT (-43.06794 -22.98509),4
...,...,...
2661,POINT (-43.08972 -22.86506),2661
2662,POINT (-43.08406 -22.86502),2662
2663,POINT (-43.08594 -22.86382),2663
2664,POINT (-43.08354 -22.86384),2664


### Salvar o arquivo

In [ ]:
amostra_aleatoria_coord_100m.to_csv('amostra_aleatoria_coord_100m.csv',index=False, encoding = 'utf-8-sig')
files.download('amostra_aleatoria_coord_100m.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Leitura do arquivo

In [ ]:
amostra_aleatoria_coord_100m = pd.read_csv('amostra_aleatoria_coord_100m.csv'
                                                  ,sep = ','
                                                  ,decimal='.'
                                                   )

In [ ]:
amostra_aleatoria_coord_100m['geometry'] = amostra_aleatoria_coord_100m['geometry'].apply(wkt.loads)
gdf_amostra_aleatoria_coord_100m = geopandas.GeoDataFrame(amostra_aleatoria_coord_100m, crs='epsg:4326')

# Inserção das ocorrências

Colocar nas coordenadas criadas em 50m com as informações das ocorrências.

In [ ]:
ocorrencias = pd.read_csv('ocorrencias.csv'
                     ,sep = ','
                     ,decimal='.'
                      )

In [ ]:
gdf_ocorrencias = geopandas.GeoDataFrame(
    ocorrencias, geometry=geopandas.points_from_xy(x = ocorrencias.lon_ocr, y = ocorrencias.lat_ocr))

## Limpeza para pontos apenas dentro do município de Niterói

In [ ]:
pts_c_ocorrencia = Clear_Outbounds_Points_in_Polygon(niteroi,gdf_ocorrencias.geometry)

In [ ]:
df_ocorrencia = ocorrencias.loc[ocorrencias['geometry'].isin(pts_c_ocorrencia)]

In [ ]:
df_ocorrencia = df_ocorrencia.reset_index(drop=True)

## Atribuição de ocorrências a n_coord

Então aqui entrará a N_coord, se não houver n_coord estipulada em menos de 50 metros cria-se uma nova n_coord.

 Verificar o ponto com ocorrência mais próximo de outra ocorrência, se tiver em menos de um raio, assumem o mesmo ponto

### Verificação de n_coords pré existentes

Atribui a ocorrência as coordenadas já numeradas caso já exista um ponto em um raio de 50m próximo.

In [ ]:
#df_ocorrencia_com_n_coord_e_dist = atribuir_n_coord_e_dist(df_ocorrencia, gdf_amostra_aleatoria_coord_100m)
df_ocorrencia_com_n_coord_e_dist = atribuir_n_coord_e_dist(df_ocorrencia, gdf_union_ncoord) # Aqui ative apenas se estiver tendo que fazer o looping

In [ ]:
_ocorrencia_com_n_coord = df_ocorrencia_com_n_coord_e_dist.drop(
    df_ocorrencia_com_n_coord_e_dist[(df_ocorrencia_com_n_coord_e_dist.loc[:,'dist_to_nearest' ] >= 50)].index
    ).reset_index(drop=True)


## Verificação de ocorrências, sem n_coord, que sejam próximas

In [ ]:
_ocorrencia_s_n_coord = df_ocorrencia_com_n_coord_e_dist.drop(
    df_ocorrencia_com_n_coord_e_dist[(df_ocorrencia_com_n_coord_e_dist.loc[:,'dist_to_nearest' ] < 50)].index
    ).reset_index(drop=True)

In [ ]:
df_aux = copy.deepcopy(_ocorrencia_s_n_coord[['id_solicitacao',	'id_tempo',	'lat_ocr',	'lon_ocr',	'geometry']])
df_aux['Ponto_mudanca'] = np.nan
df_aux['Distancia_minima'] = np.nan

linhas = df_aux.shape[0] -1

for i in range(linhas):

    coord_i = (df_aux.at[i,'geometry'].x, df_aux.at[i,'geometry'].y)

    for j in list(range(i+1, linhas+1)):

      coord_j = (df_aux.at[j,'geometry'].x, df_aux.at[j,'geometry'].y)

      distancia = hs.haversine(coord_i,
             coord_j,
             unit=Unit.METERS)

      if   (~np.isnan(df_aux.at[j,'Distancia_minima']) and df_aux.at[j,'Distancia_minima']  >  distancia ) or (np.isnan(df_aux.at[j,'Distancia_minima']) and int(distancia) < 50 ):

        df_aux.at[j,'Distancia_minima'] = distancia
        df_aux.at[j,'Ponto_mudanca'] = df_aux.at[i, 'geometry']

In [ ]:
# Coordenadas únicas ou fonte
coord_ocorr = geopandas.GeoDataFrame( df_aux.drop(
    df_aux[(~np.isnan(df_aux.loc[:,'Distancia_minima']))].index
    ).reset_index(drop=True), geometry='geometry', crs='EPSG:4326')

In [ ]:
#N_coord_inf = len(amostra_aleatoria_coord_100m)
N_coord_inf = len(gdf_union_ncoord) # Aqui ative apenas se estiver tendo que fazer o looping
N_coord_sup = N_coord_inf+ len(coord_ocorr)

In [ ]:
 coord_ocorr['n_coord'] = range(N_coord_inf,N_coord_sup)

## Atualização tabela n_coord

In [ ]:
#gdf_union_ncoord = pd.concat([gdf_amostra_aleatoria_coord_100m, coord_ocorr[['geometry', 'n_coord']]], ignore_index=True)
gdf_union_ncoord = pd.concat([gdf_union_ncoord, coord_ocorr[['geometry', 'n_coord']]], ignore_index=True) # Aqui ative apenas se estiver tendo que fazer o looping

### Atribuição de n_coord a ocorrência

Se ainda houverem atribuições com mais de 50m refazer desde 'atribuição de ocorrencias a n_coord'. Pois o que pode ter ocorrido, uma coordenada que foi atribuida, também possui outra coordenada ligada a ela.

In [ ]:
df_ocorrencia_com_n_coord_e_dist_2 = atribuir_n_coord_e_dist(df_ocorrencia, gdf_union_ncoord)

In [ ]:
df_ocorrencia_com_n_coord_e_dist_2

,id_solicitacao,id_tempo,lat_ocr,lon_ocr,geometry,dist_to_nearest,n_coord
0,350118,11049888,-22.897553,-43.117862,POINT (-43.11786 -22.89755),0.0,2666.0
1,340118,11049888,-22.897553,-43.117862,POINT (-43.11786 -22.89755),0.0,2666.0
2,60118,11049661,-22.897599,-43.120279,POINT (-43.12028 -22.89760),0.0,2667.0
3,320118,11053981,-22.898297,-43.116643,POINT (-43.11664 -22.89830),31.195679,1732.0
4,420118,11053981,-22.897709,-43.116544,POINT (-43.11654 -22.89771),0.0,2668.0
...,...,...,...,...,...,...,...
1286,2341220,12602072,-22.883235,-43.068483,POINT (-43.06848 -22.88323),43.083103,2722.0
1287,2351220,12602158,-22.911041,-43.068733,POINT (-43.06873 -22.91104),0.0,3124.0
1288,2531220,12607928,-22.896562,-43.061014,POINT (-43.06101 -22.89656),0.0,2924.0
1289,2611220,12609378,-22.906104,-43.076415,POINT (-43.07642 -22.90610),47.64853,2752.0


# Salvar arquivos

- Informação das coordenadas (geometry, n_coord)

- Informação da ocorrência ( id_tempo,id_solicitacao,geometry, n_coord)

In [ ]:
gdf_union_ncoord.to_csv('coordenadas_mapeadas.csv',index=False, encoding = 'utf-8-sig')
files.download('coordenadas_mapeadas.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
n_coord_ocorrencias = df_ocorrencia_com_n_coord_e_dist_2[['id_tempo', 'n_coord']].drop_duplicates()

In [ ]:
n_coord_ocorrencias.to_csv('amostra_aleatoria_coord_100m.csv',index=False, encoding = 'utf-8-sig')
files.download('amostra_aleatoria_coord_100m.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>